# **Language Ablation**

**Question:** does adding training data from *related* languages improve Tagalog grapheme-to-phoneme (G2P) conversion — and does *how related* they are matter?

We train three Transformer G2P models that are identical in every way (architecture, hyperparameters, seed) **except for their training data**:

| Set | Training languages |
|---|---|
| `ph` | Tagalog + 7 other Philippine languages |
| `austro` | Tagalog + non-Philippine Austronesian languages (Indonesian, Malay, Acehnese, …) |
| `all` | Tagalog + both groups combined |

All three are evaluated on the **same held-out Tagalog test set** using phoneme error rate (PER) and word error rate (WER). If genetic proximity drives cross-lingual transfer, the Philippine mix should beat the more distantly related Austronesian mix.

## Setup

This notebook runs on Colab: clone a fresh copy of the repo, move into `notebooks/` (all paths below are relative to it), and import the project modules from `src/` — data prep (`prep_data`), beam-search decoding (`decode`), the Transformer/training utilities (`train`), and temperature-based language sampling (`temperature`).

In [29]:
%cd /content
!rm -rf /content/Hunger

import os
REPO_DIR = "/content/Hunger"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/OutForMilks/Hunger {REPO_DIR}
%cd {REPO_DIR}
!pwd

/content
Cloning into '/content/Hunger'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 193 (delta 78), reused 152 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 1.15 MiB | 4.16 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/Hunger
/content/Hunger


In [30]:
%cd notebooks

/content/Hunger/notebooks


In [32]:
import csv
import pandas as pd
import os
from pathlib import Path
import glob

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler

import sys
parent_dir = str(Path.cwd().parent)
sys.path.append(parent_dir)

from src.prep_data import *
from src.decode import *
from src.train import *
from src.temperature import *

## Cleaning & Organizing

Load the raw WikiPron TSVs (one file per language) into dataframes. Each row is a `(text, phonetic)` pair — a spelled word and its space-separated IPA transcription. Every row gets tagged with its `language` (taken from the filename) and its ablation `group` (`philippine` or `austronesian`). Duplicate and null rows are dropped later, once the Tagalog train split has been mixed in.

In [ ]:
input_dir_ph = Path("./wikipron/filipino")
input_dir_austro = Path("./wikipron/austronesian")

# The two ablation groups: Philippine languages vs. other (non-Philippine) Austronesian languages.
# "west makian" has a space, but the file on disk is west_makian_*.tsv, so convert_split
# skips it later ("[skip] ... not found") -- the austro set effectively has 6 aux languages.
ph_languages = ["bikolano", "cebuano", "hiligaynon", "ilocano", "kapampangan", "pangasinan", "tagalog", "waray"]
austro_languages = ["acehnese", "balinese", "iban", "indonesian", "makasar", "malay", "west makian"]
# tagalog is already in ph_languages, so the "all" set lists it twice -- convert_split
# writes its training pairs twice (17220 -> 34440 in the training log), i.e. tagalog
# is upweighted ~2x in the "all" mix before temperature sampling.
all_languages = ph_languages + austro_languages + ["tagalog"]
col_names = ["text", "phonetic", "language", "group"]

In [34]:
ph_files = glob.glob("../wikipron/filipino/*.tsv")
austro_files = glob.glob("../wikipron/austronesian/*.tsv")

In [ ]:
temp_ph = []
temp_austro = []

# One TSV per language: read it and tag every row with its language
# (from the filename stem) and its ablation group.
for filename in ph_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "philippine"
    temp_ph.append(temp_df)

for filename in austro_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "austronesian"
    temp_austro.append(temp_df)

ph_df = pd.concat(temp_ph, ignore_index=True)
austro_df = pd.concat(temp_austro, ignore_index=True)
all_df = pd.concat(temp_austro + temp_ph, ignore_index=True)

In [36]:
print(ph_df.head())
print("\n")
print(austro_df.head())

print("\n")
print(ph_df["language"].unique())
print("\n")
print(austro_df["language"].unique())
print("\n")
print(all_df["language"].unique())

         text             phonetic    language       group
0      Abello        ʔ a b e l j o  hiligaynon  philippine
1     Aguilar        ʔ a ɡ i l a ɾ  hiligaynon  philippine
2      Arnaiz      ʔ a ɾ n a ʔ i s  hiligaynon  philippine
3     Bacolod        b a k o l o d  hiligaynon  philippine
4  Bacolodnon  b a k o l o d n o n  hiligaynon  philippine


      text      phonetic  language         group
0     Acèh       a c ɛ h  acehnese  austronesian
1  Aleuhat  a l ɯ h a t̠  acehnese  austronesian
2    abang       a b a ŋ  acehnese  austronesian
3     adèe      a d ɛ ə̯  acehnese  austronesian
4    aleue      a l ɯ ə̯  acehnese  austronesian


['hiligaynon' 'ilocano' 'waray' 'cebuano' 'kapampangan' 'pangasinan'
 'bikolano']


['acehnese' 'west_makian' 'makasar' 'balinese' 'indonesian' 'iban' 'malay']


['acehnese' 'west_makian' 'makasar' 'balinese' 'indonesian' 'iban' 'malay'
 'hiligaynon' 'ilocano' 'waray' 'cebuano' 'kapampangan' 'pangasinan'
 'bikolano']


## Splitting

Tagalog is the *target* language, so it's the only one that gets a proper **70/15/15 train/dev/test split**. Only its train portion is mixed into the three training sets — dev and test stay held out, so every model is scored on Tagalog words none of them ever saw.

The auxiliary languages are used in full (no held-out portion needed — they only serve as extra training signal). Each language is written to its own `{lang}_train.tsv`, and `convert_split` then assembles each ablation set into the parallel `train.src` / `train.tgt` / `train.lang` files the training loop reads.

In [37]:
input_dir_tgl = Path("../wikipron/tagalog.tsv")

df_tgl = pd.read_csv(input_dir_tgl, sep='\t', names=col_names, header=None)
df_tgl["language"] = "tagalog"
df_tgl["group"] = "philippine"

In [ ]:
from sklearn.model_selection import train_test_split

# 70/15/15 split: peel off 15% for test, then 17.65% of the remaining 85% (~15% overall) for dev
train_dev_tgl, test_tgl = train_test_split(df_tgl, test_size=0.15, random_state=42)
train_tgl, dev_tgl = train_test_split(train_dev_tgl, test_size=0.1765, random_state=42)

ph_aux_df = ph_df.copy()
austro_aux_df = austro_df.copy()

# Each training mix = the tagalog train split + its auxiliary language group
tagalog_only_df = train_tgl.copy()
ph_df = pd.concat([train_tgl, ph_aux_df], ignore_index=True)
austro_df = pd.concat([train_tgl, austro_aux_df], ignore_index=True)
all_df = pd.concat([train_tgl, ph_aux_df, austro_aux_df], ignore_index=True)

# Clean only now, so duplicates between the tagalog split and the aux data are caught too
for d in [tagalog_only_df, ph_df, austro_df, all_df]:
    d.drop_duplicates(subset=["text", "phonetic"], inplace=True)
    d.dropna(inplace=True)

In [39]:
# ph_df.to_csv("ablation/input/ph_train.tsv", sep="\t", index=False, header=False)
# austro_df.to_csv("ablation/input/austro_train.tsv", sep="\t", index=False, header=False)
# all_df.to_csv("ablation/input/all_train.tsv", sep="\t", index=False, header=False)
# dev_tgl.to_csv("ablation/input/tgl_dev.tsv", sep="\t", index=False, header=False)
# test_tgl.to_csv("ablation/input/tgl_test.tsv", sep="\t", index=False, header=False)

In [41]:
input_dir = Path("ablation/input")
input_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("ablation/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# Pool all training data, dedupe, then write one {lang}_train.tsv per language
# so convert_split can assemble any subset of languages into a training set.
combined_df = pd.concat(
    [ph_aux_df, austro_aux_df, train_tgl], ignore_index=True
).drop_duplicates(subset=["text", "phonetic"])

for lang, group_df in combined_df.groupby("language"):
    group_df[["text", "phonetic"]].to_csv(
        input_dir / f"{lang}_train.tsv", sep="\t", index=False, header=False
    )
    print(f"{lang}: {len(group_df)} pairs")

# The held-out tagalog dev/test splits -- every model is evaluated on these
dev_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_dev.tsv", sep="\t", index=False, header=False)
test_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_test.tsv", sep="\t", index=False, header=False)


acehnese: 265 pairs
balinese: 290 pairs
bikolano: 4178 pairs
cebuano: 3684 pairs
hiligaynon: 473 pairs
iban: 457 pairs
ilocano: 992 pairs
indonesian: 18225 pairs
kapampangan: 902 pairs
makasar: 807 pairs
malay: 4321 pairs
pangasinan: 168 pairs
tagalog: 17220 pairs
waray: 302 pairs
west_makian: 775 pairs


In [42]:
# Assemble the three ablation training sets: each becomes parallel
# train.src (graphemes) / train.tgt (phonemes) / train.lang (language tag) files.
sets = {
    "ph": ph_languages,
    "austro": austro_languages,
    "all": all_languages,
}

for set_name, langs in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

# Dev/test are tagalog-only: shared across all three models
for split in ["dev", "test"]:
    out_dir = f"ablation/splits/tgl_{split}"
    print(f"Split: {split}")
    convert_split(input_dir, out_dir, ["tagalog"], split)

Set: ph
/content/Hunger/notebooks
  bikolano train: 4178 pairs
  cebuano train: 3684 pairs
  hiligaynon train: 473 pairs
  ilocano train: 992 pairs
  kapampangan train: 902 pairs
  pangasinan train: 168 pairs
  tagalog train: 17220 pairs
  waray train: 302 pairs
  -> wrote 27919 lines to train.{src,tgt,lang}
Set: austro
/content/Hunger/notebooks
  acehnese train: 265 pairs
  balinese train: 290 pairs
  iban train: 457 pairs
  indonesian train: 18225 pairs
  makasar train: 807 pairs
  malay train: 4321 pairs
  [skip] ablation/input/west makian_train.tsv not found
  -> wrote 24365 lines to train.{src,tgt,lang}
Set: all
/content/Hunger/notebooks
  bikolano train: 4178 pairs
  cebuano train: 3684 pairs
  hiligaynon train: 473 pairs
  ilocano train: 992 pairs
  kapampangan train: 902 pairs
  pangasinan train: 168 pairs
  tagalog train: 17220 pairs
  waray train: 302 pairs
  acehnese train: 265 pairs
  balinese train: 290 pairs
  iban train: 457 pairs
  indonesian train: 18225 pairs
  makasa

## Train

One model per ablation set, all sharing the exact same configuration and seed — the training data is the only variable, so any difference in test scores is attributable to the language mix.

The model is a standard encoder–decoder Transformer (6 layers, `d_model=512`, 8 heads, FF 2048) trained with the classic *Attention Is All You Need* recipe: Adam + Noam LR warmup, label smoothing 0.1, dropout 0.1. Each run is 1000 steps at batch size 256.

Because the language sizes are wildly imbalanced (168 Pangasinan pairs vs. ~18k Indonesian), batches are drawn with **temperature sampling** (T=5): the natural language distribution `p` is flattened into `q`, so low-resource languages are seen far more often than their raw share — the per-language `p → q` tables are printed at the start of each run.

In [43]:
DATA = Path("ablation/splits")
OUTPUT = Path("./models")
#seed is a var
STEPS = 1000
SAVE_STEP = 1000

# Transformer-base-style architecture (Vaswani et al. 2017)
BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

# Regularization + Noam schedule warmup steps
DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

# Saved into every checkpoint so a .pt file is self-describing
CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 100

cuda


In [44]:
sets = ["ph", "austro", "all"]
seed = 42

if xla:

    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
else:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

# Train one model per ablation set; everything below is identical across sets
# except data_path/output_path.
for set_name in sets:
    print(f"\nTraining set: {set_name}")
    DATA = Path(f"ablation/splits/{set_name}")
    OUTPUT = Path(f"./models/{set_name}")
    os.makedirs(OUTPUT, exist_ok=True)

    config_1 = dict(CONFIG)
    config_1["seed"] = seed
    config_1["data_path"] = str(DATA)
    config_1["output_path"] = str(OUTPUT)

    # Re-seed per run so each model trains from the same init regardless of set order
    torch.manual_seed(seed)
    device = torch.device(DEVICE)

    # Vocab is built from this set's training data only (each set has its own
    # grapheme/phoneme inventory), then saved alongside the checkpoints.
    src_vocab, tgt_vocab = build_vocabs(
        os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
    save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

    train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                            os.path.join(DATA, "train.tgt"),
                            src_vocab, tgt_vocab)
    # Fixed max lengths let every batch have the same shape (needed for XLA,
    # harmless on GPU).
    max_src, max_tgt = compute_max_lengths(train_ds)
    print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
            f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

    # Temperature sampling (T=5) flattens the language imbalance: low-resource
    # languages get sampled far above their natural frequency.
    weights = temperature_sampling_weights(
        os.path.join(DATA, "train.lang"), temperature=5.0
    )
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                        collate_fn=make_fixed_collate(max_src, max_tgt),
                        drop_last=True)
    if xla:
        loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

    model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                        n_heads=NUM_HEADS, d_ff=FF_SIZE,
                        n_layers=N_LAYERS, dropout=DROPOUT,
                        pad_id=PAD_ID).to(device)
    # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
    opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
    criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

    model.train()
    data_iter = infinite_loader(loader)
    for step in range(1, STEPS + 1):
        batch = next(data_iter)
        if xla:
            src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
        else:
            src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

        # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
        logits = model(src, tgt_in, src_pad, tgt_pad)
        loss = criterion(logits, tgt_out)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

        if step % log_step == 0:
            # .item() forces a host sync; only do it at log intervals on TPU.
            print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                    f"lr {opt.param_groups[0]['lr']:.2e}")

        if step % SAVE_STEP == 0 or step == STEPS:
            # Checkpoint bundles weights + config + both vocabs, so evaluation
            # can rebuild the model from the .pt file alone.
            ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
            payload = {"model": model.state_dict(),
                        "config": config_1,
                        "src_vocab": src_vocab.to_dict(),
                        "tgt_vocab": tgt_vocab.to_dict()}
            if xla:
                xm.save(payload, ckpt)   # moves tensors to CPU; master-only
            else:
                torch.save(payload, ckpt)
            print(f"  saved {ckpt}")



Training set: ph
src vocab=74 tgt vocab=67 examples=27919 | fixed shapes: src=48 tgt=32
Language distribution (natural -> temperature-scaled):
  bikolano     n=  4178  p=0.1496  q=0.1526
  cebuano      n=  3684  p=0.1320  q=0.1488
  hiligaynon   n=   473  p=0.0169  q=0.0987
  ilocano      n=   992  p=0.0355  q=0.1145
  kapampangan  n=   902  p=0.0323  q=0.1123
  pangasinan   n=   168  p=0.0060  q=0.0803
  tagalog      n= 17220  p=0.6168  q=0.2026
  waray        n=   302  p=0.0108  q=0.0902
step     100/1000  loss 2.6184  lr 1.75e-05
step     200/1000  loss 2.1317  lr 3.49e-05
step     300/1000  loss 1.7635  lr 5.24e-05
step     400/1000  loss 1.6157  lr 6.99e-05
step     500/1000  loss 1.4923  lr 8.73e-05
step     600/1000  loss 1.3958  lr 1.05e-04
step     700/1000  loss 1.3424  lr 1.22e-04
step     800/1000  loss 1.2535  lr 1.40e-04
step     900/1000  loss 1.1958  lr 1.57e-04
step    1000/1000  loss 1.1929  lr 1.75e-04
  saved models/ph/ckpt_1000.pt

Training set: austro
src vocab=7

In [50]:
# for saving the .pt model stuff from colab if needed
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# !mkdir -p /content/drive/MyDrive/Hunger_checkpoints
# !cp -r /content/Hunger/notebooks/models/* /content/drive/MyDrive/Hunger_checkpoints/

Mounted at /content/drive


## Evaluation

Each model decodes the held-out Tagalog test set (4,245 words) with beam search (`beam=5`) and is scored on:

- **PER** (phoneme error rate) — edit distance between predicted and reference phoneme sequences, normalized by reference length; and
- **WER** (word error rate) — fraction of words whose transcription isn't an exact match.

Lower is better for both. Since the models differ only in training mix, these two numbers *are* the ablation result.

In [51]:
from src.evaluation import *

device = torch.device(DEVICE)
print(device)

# Score every ablation model on the same held-out tagalog test set
results = {}
for set_name in ["ph", "austro", "all"]:
    print(f"\nSet: {set_name}")
    ckpt_path = f"./models/{set_name}/ckpt_{STEPS}.pt"

    per, wer, refs, hyps = evaluate_checkpoint(
        model_paths=[ckpt_path],
        src_path="ablation/splits/tgl_test/test.src",
        tgt_path="ablation/splits/tgl_test/test.tgt",
        device=device,
        beam=5,
    )
    results[set_name] = {"PER": per, "WER": wer}
    print(f"{set_name}: PER={per:.4f}  WER={wer:.4f}")

print("\nSummary:")
for name, r in results.items():
    print(f"{name}  PER={r['PER']:.4f}  WER={r['WER']:.4f}")

cuda

Set: ph
  decoded 200/4245
  decoded 400/4245
  decoded 600/4245
  decoded 800/4245
  decoded 1000/4245
  decoded 1200/4245
  decoded 1400/4245
  decoded 1600/4245
  decoded 1800/4245
  decoded 2000/4245
  decoded 2200/4245
  decoded 2400/4245
  decoded 2600/4245
  decoded 2800/4245
  decoded 3000/4245
  decoded 3200/4245
  decoded 3400/4245
  decoded 3600/4245
  decoded 3800/4245
  decoded 4000/4245
  decoded 4200/4245
ph: PER=0.2004  WER=0.5559

Set: austro
  decoded 200/4245
  decoded 400/4245
  decoded 600/4245
  decoded 800/4245
  decoded 1000/4245
  decoded 1200/4245
  decoded 1400/4245
  decoded 1600/4245
  decoded 1800/4245
  decoded 2000/4245
  decoded 2200/4245
  decoded 2400/4245
  decoded 2600/4245
  decoded 2800/4245
  decoded 3000/4245
  decoded 3200/4245
  decoded 3400/4245
  decoded 3600/4245
  decoded 3800/4245
  decoded 4000/4245
  decoded 4200/4245
austro: PER=0.4438  WER=0.8912

Set: all
  decoded 200/4245
  decoded 400/4245
  decoded 600/4245
  decoded 800/42

### Results & takeaways

| Training set | PER ↓ | WER ↓ |
|---|---|---|
| `ph` (Tagalog + Philippine) | **0.2004** | **0.5559** |
| `austro` (Tagalog + other Austronesian) | 0.4438 | 0.8912 |
| `all` (everything) | 0.2222 | 0.6040 |



## More ablation lol

### Cleaning

In [ ]:
col_names = ["text", "phonetic", "language", "group"]

df_eng = pd.read_csv("./wikipron/english.tsv", sep='\t', names=col_names, header=None)
df_eng["language"] = "english"
df_eng["group"] = "english"


df_spa = pd.read_csv("./wikipron/spanish.tsv", sep='\t', names=col_names, header=None)
df_spa["language"] = "spanish"
df_spa["group"] = "spanish"

df_eng.drop_duplicates().dropna()
df_spa.drop_duplicates().dropna()

df_eng = pd.concat([ph_df, df_eng], ignore_index=True)
df_spa = pd.concat([ph_df, df_spa], ignore_index=True)

df_eng.to_csv("ablation/input/english_train.tsv", sep="\t", index=False, header=False)
df_spa.to_csv("ablation/input/spanish_train.tsv", sep="\t", index=False, header=False)

In [ ]:
sets = {"english", "spanish"}

for set_name in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

### Train

In [ ]:
DATA = Path("ablation/splits")
OUTPUT = Path("./models")
#seed is a var
STEPS = 1000
SAVE_STEP = 1000

# Transformer-base-style architecture (Vaswani et al. 2017)
BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

# Regularization + Noam schedule warmup steps
DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

# Saved into every checkpoint so a .pt file is self-describing
CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 100

In [ ]:
temps = ["english", "spanish"]
seed = 42

if xla:

    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
else:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

# Train one model per ablation set; everything below is identical across sets
# except data_path/output_path.
for set_name in sets:
    print(f"\nTraining set: {set_name}")
    DATA = Path(f"ablation/splits/{set_name}")
    OUTPUT = Path(f"./models/{set_name}")
    os.makedirs(OUTPUT, exist_ok=True)

    config_1 = dict(CONFIG)
    config_1["seed"] = seed
    config_1["data_path"] = str(DATA)
    config_1["output_path"] = str(OUTPUT)

    # Re-seed per run so each model trains from the same init regardless of set order
    torch.manual_seed(seed)
    device = torch.device(DEVICE)

    # Vocab is built from this set's training data only (each set has its own
    # grapheme/phoneme inventory), then saved alongside the checkpoints.
    src_vocab, tgt_vocab = build_vocabs(
        os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
    save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

    train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                            os.path.join(DATA, "train.tgt"),
                            src_vocab, tgt_vocab)
    # Fixed max lengths let every batch have the same shape (needed for XLA,
    # harmless on GPU).
    max_src, max_tgt = compute_max_lengths(train_ds)
    print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
            f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

    # Temperature sampling (T=5) flattens the language imbalance: low-resource
    # languages get sampled far above their natural frequency.
    weights = temperature_sampling_weights(
        os.path.join(DATA, "train.lang"), temperature=5.0
    )
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                        collate_fn=make_fixed_collate(max_src, max_tgt),
                        drop_last=True)
    if xla:
        loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

    model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                        n_heads=NUM_HEADS, d_ff=FF_SIZE,
                        n_layers=N_LAYERS, dropout=DROPOUT,
                        pad_id=PAD_ID).to(device)
    # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
    opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
    criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

    model.train()
    data_iter = infinite_loader(loader)
    for step in range(1, STEPS + 1):
        batch = next(data_iter)
        if xla:
            src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
        else:
            src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

        # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
        logits = model(src, tgt_in, src_pad, tgt_pad)
        loss = criterion(logits, tgt_out)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

        if step % log_step == 0:
            # .item() forces a host sync; only do it at log intervals on TPU.
            print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                    f"lr {opt.param_groups[0]['lr']:.2e}")

        if step % SAVE_STEP == 0 or step == STEPS:
            # Checkpoint bundles weights + config + both vocabs, so evaluation
            # can rebuild the model from the .pt file alone.
            ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
            payload = {"model": model.state_dict(),
                        "config": config_1,
                        "src_vocab": src_vocab.to_dict(),
                        "tgt_vocab": tgt_vocab.to_dict()}
            if xla:
                xm.save(payload, ckpt)   # moves tensors to CPU; master-only
            else:
                torch.save(payload, ckpt)
            print(f"  saved {ckpt}")


### Evaluation

In [ ]:
device = torch.device(DEVICE)
print(device)

# Score every ablation model on the same held-out tagalog test set
results = {}
for set_name in ["english", "spanish"]:
    print(f"\nSet: {set_name}")
    ckpt_path = f"./models/{set_name}/ckpt_{STEPS}.pt"

    per, wer, refs, hyps = evaluate_checkpoint(
        model_paths=[ckpt_path],
        src_path="ablation/splits/tgl_test/test.src",
        tgt_path="ablation/splits/tgl_test/test.tgt",
        device=device,
        beam=5,
    )
    results[set_name] = {"PER": per, "WER": wer}
    print(f"{set_name}: PER={per:.4f}  WER={wer:.4f}")

print("\nSummary:")
for name, r in results.items():
    print(f"{name}  PER={r['PER']:.4f}  WER={r['WER']:.4f}")

In [ ]:
best_set = min(results, key=lambda name: results[name]["PER"])

## Temperature Tuning

Since the Philippine language set performed the best on the tagalog test set we will use that for temp tuning 

In [ ]:
DATA = Path(f"ablation/splits/{best_set}")
OUTPUT = Path(f"./models/{best_set}")
#seed is a var
STEPS = 10000
SAVE_STEP = 10000

# Transformer-base-style architecture (Vaswani et al. 2017)
BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

# Regularization + Noam schedule warmup steps
DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

# Saved into every checkpoint so a .pt file is self-describing
CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 1000

In [ ]:
seed = 42
temps = [1.0, 2.0, 3.0, 5.0, 7.0]
temp_results = {}

if xla:

    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
else:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

# Train one model per ablation set; everything below is identical across sets
# except data_path/output_path.
for temperature in temps:
    print(f"\nTemperature: {temperature}")
    DATA = Path(f"ablation/splits/{best_set}")
    OUTPUT = Path(f"./models/{best_set}")
    os.makedirs(OUTPUT, exist_ok=True)

    config_1 = dict(CONFIG)
    config_1["seed"] = seed
    config_1["data_path"] = str(DATA)
    config_1["output_path"] = str(OUTPUT)
    config_1["temperature"] = temperature

    # Re-seed per run so each model trains from the same init regardless of set order
    torch.manual_seed(seed)
    device = torch.device(DEVICE)

    # Vocab is built from this set's training data only (each set has its own
    # grapheme/phoneme inventory), then saved alongside the checkpoints.
    src_vocab, tgt_vocab = build_vocabs(
        os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
    save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

    train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                            os.path.join(DATA, "train.tgt"),
                            src_vocab, tgt_vocab)
    # Fixed max lengths let every batch have the same shape (needed for XLA,
    # harmless on GPU).
    max_src, max_tgt = compute_max_lengths(train_ds)
    print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
            f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

    # Temperature sampling (T=5) flattens the language imbalance: low-resource
    # languages get sampled far above their natural frequency.
    weights = temperature_sampling_weights(
        os.path.join(DATA, "train.lang"), temperature=temperature
    )
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                        collate_fn=make_fixed_collate(max_src, max_tgt),
                        drop_last=True)
    if xla:
        loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

    model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                        n_heads=NUM_HEADS, d_ff=FF_SIZE,
                        n_layers=N_LAYERS, dropout=DROPOUT,
                        pad_id=PAD_ID).to(device)
    # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
    opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
    criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

    model.train()
    data_iter = infinite_loader(loader)
    for step in range(1, STEPS + 1):
        batch = next(data_iter)
        if xla:
            src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
        else:
            src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

        # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
        logits = model(src, tgt_in, src_pad, tgt_pad)
        loss = criterion(logits, tgt_out)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

        if step % log_step == 0:
            # .item() forces a host sync; only do it at log intervals on TPU.
            print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                    f"lr {opt.param_groups[0]['lr']:.2e}")

        if step % SAVE_STEP == 0 or step == STEPS:
            # Checkpoint bundles weights + config + both vocabs, so evaluation
            # can rebuild the model from the .pt file alone.
            ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
            payload = {"model": model.state_dict(),
                        "config": config_1,
                        "src_vocab": src_vocab.to_dict(),
                        "tgt_vocab": tgt_vocab.to_dict()}
            if xla:
                xm.save(payload, ckpt)   # moves tensors to CPU; master-only
            else:
                torch.save(payload, ckpt)
            print(f"  saved {ckpt}")
    
    del model, opt, sched, criterion, loader, train_ds
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    ckpt_path = os.path.join(OUTPUT, f"ckpt_{STEPS}.pt")
    per, wer, _, _ = evaluate_checkpoint(
        model_paths=[ckpt_path],
        src_path="ablation/splits/tgl_dev/dev.src",
        tgt_path="ablation/splits/tgl_dev/dev.tgt",
        device=device, beam=5,
    )
    temp_results[temperature] = {"PER": per, "WER": wer}
    print(f"T={temperature}: dev PER={per:.4f}  WER={wer:.4f}")


In [ ]:
print("\nTemperature summary:")
for t, r in sorted(temp_results.items(), key=lambda x: x[1]["PER"]):
    print(f"T={t}: PER={r['PER']:.4f}  WER={r['WER']:.4f}")

best_temp = min(temp_results, key=lambda t: temp_results[t]["PER"])
print(f"\nBest temperature: {best_temp}")